In [ ]:
!pip install imbalanced-learn -q

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 5000

# Customer demographics
customer_id = np.arange(1, N + 1)
tenure_months = np.random.randint(1, 60, N)
age = np.random.randint(18, 70, N)
gender = np.random.choice(["Male", "Female"], N)
city_tier = np.random.choice([1, 2, 3], N, p=[0.3, 0.4, 0.3])

# Behavioral features
num_orders = np.random.poisson(lam=tenure_months * 0.6 + 2, size=N).clip(1, 200)
avg_order_value = np.random.lognormal(mean=4.2, sigma=0.8, size=N).round(2)
total_spend = (num_orders * avg_order_value).round(2)
days_since_last_order = np.random.exponential(scale=30, size=N).astype(int).clip(1, 365)
num_returns = np.random.poisson(lam=num_orders * 0.05, size=N)
return_rate = (num_returns / num_orders).round(4)

# Engagement features
num_sessions = np.random.poisson(lam=tenure_months * 4, size=N).clip(1, 500)
avg_session_duration_min = np.random.exponential(scale=8, size=N).round(2).clip(0.5, 60)
email_open_rate = np.random.beta(2, 5, N).round(4)
app_used = np.random.choice([0, 1], N, p=[0.3, 0.7])
support_tickets = np.random.poisson(lam=0.8, size=N)
satisfaction_score = np.random.choice([1, 2, 3, 4, 5], N, p=[0.05, 0.1, 0.2, 0.35, 0.3])
payment_method = np.random.choice(
    ["Credit Card", "Debit Card", "UPI", "Wallet", "Net Banking"],
    N, p=[0.3, 0.25, 0.2, 0.15, 0.1]
)
category_preference = np.random.choice(
    ["Electronics", "Fashion", "Grocery", "Home", "Beauty", "Sports"],
    N, p=[0.2, 0.25, 0.15, 0.15, 0.15, 0.1]
)
discount_usage_rate = np.random.beta(3, 4, N).round(4)
coupon_redeemed = np.random.choice([0, 1], N, p=[0.45, 0.55])

# Churn probability (realistic non-linear logic)
churn_score = (
    0.3 * (days_since_last_order / 365)
    - 0.2 * (tenure_months / 60)
    - 0.15 * (satisfaction_score / 5)
    + 0.2 * return_rate
    + 0.1 * (support_tickets / 5)
    - 0.1 * email_open_rate
    - 0.05 * app_used
    + 0.1 * (city_tier == 3).astype(int)
    + np.random.normal(0, 0.1, N)
)
churn_prob = 1 / (1 + np.exp(-5 * churn_score))
churn = (churn_prob > 0.5).astype(int)

df = pd.DataFrame({
    "customer_id": customer_id,
    "tenure_months": tenure_months,
    "age": age,
    "gender": gender,
    "city_tier": city_tier,
    "num_orders": num_orders,
    "avg_order_value": avg_order_value,
    "total_spend": total_spend,
    "days_since_last_order": days_since_last_order,
    "num_returns": num_returns,
    "return_rate": return_rate,
    "num_sessions": num_sessions,
    "avg_session_duration_min": avg_session_duration_min,
    "email_open_rate": email_open_rate,
    "app_used": app_used,
    "support_tickets": support_tickets,
    "satisfaction_score": satisfaction_score,
    "payment_method": payment_method,
    "category_preference": category_preference,
    "discount_usage_rate": discount_usage_rate,
    "coupon_redeemed": coupon_redeemed,
    "churn": churn,
})

df.to_csv("/content/ecommerce_customers.csv", index=False)
print(f"Dataset saved: {df.shape}")
print(f"Churn rate: {df['churn'].mean():.2%}")

Dataset saved: (5000, 22)
Churn rate: 7.38%


In [ ]:
"""
E-Commerce Customer Churn Prediction
=====================================
Real-world ML project: predict which customers are likely to churn
so the business can take proactive retention actions.

Dataset: Synthetic but realistic e-commerce customer data (5,000 customers, 21 features)
Problem type: Binary Classification
Target: churn (1 = churned, 0 = retained)
"""

# ─────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use(Agg)  # not needed in Colab
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

PALETTE = {"churn": "#e05c5c", "retain": "#4a90d9", "neutral": "#7b8fa1"}
CHURN_COLORS = [PALETTE["retain"], PALETTE["churn"]]

# ─────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv("/content/ecommerce_customers.csv")
df_raw = df.copy()

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape          : {df.shape}")
print(f"Churn rate     : {df['churn'].mean():.2%}")
print(f"Missing values : {df.isnull().sum().sum()}")
print("\nData types:")
print(df.dtypes.value_counts())
print("\nFirst 3 rows:")
print(df.head(3).to_string())


# ─────────────────────────────────────────────
# 3. EDA — Figure 1: Overview & Target
# ─────────────────────────────────────────────
fig1, axes = plt.subplots(2, 3, figsize=(16, 10))
fig1.suptitle("EDA — Overview & Target Distribution", fontsize=16, fontweight="bold", y=0.98)

# 3a. Churn distribution (donut)
ax = axes[0, 0]
counts = df["churn"].value_counts()
wedges, texts, autotexts = ax.pie(
    counts, labels=["Retained", "Churned"],
    colors=CHURN_COLORS, autopct="%1.1f%%",
    startangle=90, wedgeprops=dict(width=0.55), pctdistance=0.75
)
for t in autotexts:
    t.set_fontsize(12); t.set_fontweight("bold")
ax.set_title("Target: Churn Distribution", fontweight="bold")

# 3b. Tenure vs churn
ax = axes[0, 1]
for c, label, color in zip([0, 1], ["Retained", "Churned"], CHURN_COLORS):
    ax.hist(df[df["churn"] == c]["tenure_months"], bins=20, alpha=0.7,
            label=label, color=color, edgecolor="white")
ax.set_xlabel("Tenure (months)"); ax.set_ylabel("Count")
ax.set_title("Tenure Distribution by Churn", fontweight="bold")
ax.legend()

# 3c. Days since last order
ax = axes[0, 2]
for c, label, color in zip([0, 1], ["Retained", "Churned"], CHURN_COLORS):
    ax.hist(df[df["churn"] == c]["days_since_last_order"].clip(0, 150),
            bins=20, alpha=0.7, label=label, color=color, edgecolor="white")
ax.set_xlabel("Days Since Last Order"); ax.set_ylabel("Count")
ax.set_title("Recency Distribution by Churn", fontweight="bold")
ax.legend()

# 3d. Satisfaction score vs churn rate
ax = axes[1, 0]
churn_by_sat = df.groupby("satisfaction_score")["churn"].mean() * 100
bars = ax.bar(churn_by_sat.index, churn_by_sat.values,
              color=[PALETTE["churn"] if v > 10 else PALETTE["retain"] for v in churn_by_sat.values],
              edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, churn_by_sat.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.1f}%",
            ha="center", va="bottom", fontsize=10)
ax.set_xlabel("Satisfaction Score"); ax.set_ylabel("Churn Rate (%)")
ax.set_title("Satisfaction Score vs Churn Rate", fontweight="bold")

# 3e. City tier vs churn rate
ax = axes[1, 1]
churn_by_tier = df.groupby("city_tier")["churn"].mean() * 100
ax.bar([f"Tier {t}" for t in churn_by_tier.index], churn_by_tier.values,
       color=[PALETTE["neutral"], PALETTE["churn"], PALETTE["retain"]],
       edgecolor="white", linewidth=1.5)
for i, val in enumerate(churn_by_tier.values):
    ax.text(i, val + 0.2, f"{val:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_xlabel("City Tier"); ax.set_ylabel("Churn Rate (%)")
ax.set_title("City Tier vs Churn Rate", fontweight="bold")

# 3f. Age distribution
ax = axes[1, 2]
for c, label, color in zip([0, 1], ["Retained", "Churned"], CHURN_COLORS):
    ax.hist(df[df["churn"] == c]["age"], bins=25, alpha=0.7,
            label=label, color=color, edgecolor="white")
ax.set_xlabel("Age"); ax.set_ylabel("Count")
ax.set_title("Age Distribution by Churn", fontweight="bold")
ax.legend()

plt.tight_layout()
fig1.savefig("/content/fig1_overview_eda.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n[✓] Figure 1 saved")


# ─────────────────────────────────────────────
# 4. EDA — Figure 2: Behavioral Patterns
# ─────────────────────────────────────────────
fig2, axes = plt.subplots(2, 3, figsize=(16, 10))
fig2.suptitle("EDA — Behavioral & Engagement Patterns", fontsize=16, fontweight="bold", y=0.98)

# 4a. Return rate boxplot
ax = axes[0, 0]
data_box = [df[df["churn"] == 0]["return_rate"], df[df["churn"] == 1]["return_rate"]]
bp = ax.boxplot(data_box, patch_artist=True, labels=["Retained", "Churned"])
for patch, color in zip(bp["boxes"], CHURN_COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel("Return Rate")
ax.set_title("Return Rate by Churn", fontweight="bold")

# 4b. Email open rate vs churn
ax = axes[0, 1]
churn_by_app = df.groupby("app_used")["churn"].mean() * 100
ax.bar(["No App", "Uses App"], churn_by_app.values,
       color=[PALETTE["churn"], PALETTE["retain"]], edgecolor="white", linewidth=1.5)
for i, val in enumerate(churn_by_app.values):
    ax.text(i, val + 0.2, f"{val:.1f}%", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("Churn Rate (%)"); ax.set_title("App Usage vs Churn Rate", fontweight="bold")

# 4c. Support tickets
ax = axes[0, 2]
churn_by_tickets = df.groupby("support_tickets")["churn"].mean().reset_index()
churn_by_tickets = churn_by_tickets[churn_by_tickets["support_tickets"] <= 5]
ax.plot(churn_by_tickets["support_tickets"], churn_by_tickets["churn"] * 100,
        marker="o", color=PALETTE["churn"], linewidth=2.5, markersize=8)
ax.fill_between(churn_by_tickets["support_tickets"],
                churn_by_tickets["churn"] * 100, alpha=0.15, color=PALETTE["churn"])
ax.set_xlabel("Support Tickets"); ax.set_ylabel("Churn Rate (%)")
ax.set_title("Support Tickets vs Churn Rate", fontweight="bold")
ax.grid(axis="y", alpha=0.4)

# 4d. Category preference churn
ax = axes[1, 0]
churn_by_cat = df.groupby("category_preference")["churn"].mean().sort_values(ascending=True) * 100
colors = [PALETTE["retain"] if v < churn_by_cat.mean() else PALETTE["churn"] for v in churn_by_cat.values]
ax.barh(churn_by_cat.index, churn_by_cat.values, color=colors, edgecolor="white")
ax.axvline(churn_by_cat.mean(), color="gray", linestyle="--", alpha=0.7, label=f"Avg: {churn_by_cat.mean():.1f}%")
ax.set_xlabel("Churn Rate (%)"); ax.set_title("Category Preference vs Churn", fontweight="bold")
ax.legend(fontsize=9)

# 4e. Payment method churn
ax = axes[1, 1]
churn_by_pay = df.groupby("payment_method")["churn"].mean().sort_values(ascending=False) * 100
ax.bar(range(len(churn_by_pay)), churn_by_pay.values,
       color=PALETTE["neutral"], edgecolor="white", linewidth=1.5)
ax.set_xticks(range(len(churn_by_pay)))
ax.set_xticklabels(churn_by_pay.index, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Churn Rate (%)"); ax.set_title("Payment Method vs Churn", fontweight="bold")

# 4f. Discount usage
ax = axes[1, 2]
df["discount_bin"] = pd.cut(df["discount_usage_rate"], bins=5,
                            labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"])
churn_by_disc = df.groupby("discount_bin", observed=True)["churn"].mean() * 100
ax.bar(range(len(churn_by_disc)), churn_by_disc.values,
       color=[PALETTE["churn"] if v > 8 else PALETTE["retain"] for v in churn_by_disc.values],
       edgecolor="white")
ax.set_xticks(range(len(churn_by_disc)))
ax.set_xticklabels(churn_by_disc.index, fontsize=9)
ax.set_xlabel("Discount Usage Rate"); ax.set_ylabel("Churn Rate (%)")
ax.set_title("Discount Usage vs Churn Rate", fontweight="bold")

plt.tight_layout()
fig2.savefig("/content/fig2_behavioral_eda.png", dpi=150, bbox_inches="tight")
plt.close()
print("[✓] Figure 2 saved")


# ─────────────────────────────────────────────
# 5. EDA — Figure 3: Correlation Heatmap
# ─────────────────────────────────────────────
fig3, ax = plt.subplots(figsize=(13, 10))
num_cols = ["tenure_months", "age", "num_orders", "avg_order_value", "total_spend",
            "days_since_last_order", "return_rate", "num_sessions", "avg_session_duration_min",
            "email_open_rate", "support_tickets", "satisfaction_score", "discount_usage_rate", "churn"]
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 8})
ax.set_title("Feature Correlation Heatmap", fontsize=15, fontweight="bold", pad=15)
plt.tight_layout()
fig3.savefig("/content/fig3_correlation.png", dpi=150, bbox_inches="tight")
plt.close()
print("[✓] Figure 3 saved")


# ─────────────────────────────────────────────
# 6. PREPROCESSING
# ─────────────────────────────────────────────
df_model = df_raw.copy()
df_model.drop(columns=["customer_id", "discount_bin"] if "discount_bin" in df_model.columns else ["customer_id"], inplace=True, errors="ignore")

le = LabelEncoder()
for col in ["gender", "payment_method", "category_preference"]:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop("churn", axis=1)
y = df_model["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_test_sc  = scaler.transform(X_test)

print(f"\n[✓] Preprocessing done")
print(f"    Train size: {X_train_sc.shape[0]} | Test size: {X_test_sc.shape[0]}")
print(f"    After SMOTE — Class 0: {(y_train_bal==0).sum()} | Class 1: {(y_train_bal==1).sum()}")


# ─────────────────────────────────────────────
# 7. MODEL TRAINING
# ─────────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, C=1.0),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                                     max_depth=4, random_state=42),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 60)
print("MODEL TRAINING & CROSS-VALIDATION")
print("=" * 60)
for name, model in models.items():
    model.fit(X_train_sc, y_train_bal)
    cv_scores = cross_val_score(model, X_train_sc, y_train_bal, cv=cv, scoring="roc_auc", n_jobs=-1)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    y_pred = model.predict(X_test_sc)
    test_auc = roc_auc_score(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    results[name] = {
        "model": model, "y_prob": y_prob, "y_pred": y_pred,
        "cv_auc": cv_scores.mean(), "cv_std": cv_scores.std(),
        "test_auc": test_auc, "avg_precision": ap,
    }
    print(f"\n  {name}")
    print(f"    CV AUC  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"    Test AUC: {test_auc:.4f}")
    print(f"    Avg Prec: {ap:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))

best_name = max(results, key=lambda k: results[k]["test_auc"])
best = results[best_name]
print(f"\n[★] Best model: {best_name} (AUC = {best['test_auc']:.4f})")


# ─────────────────────────────────────────────
# 8. EVALUATION — Figure 4
# ─────────────────────────────────────────────
fig4, axes = plt.subplots(2, 3, figsize=(18, 11))
fig4.suptitle("Model Evaluation Dashboard", fontsize=16, fontweight="bold", y=0.99)

colors_map = {"Logistic Regression": "#4a90d9", "Random Forest": "#2ecc71", "Gradient Boosting": "#e67e22"}

# 8a. ROC curves
ax = axes[0, 0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    ax.plot(fpr, tpr, linewidth=2.5, label=f"{name} (AUC={res['test_auc']:.3f})",
            color=colors_map[name])
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 8b. Precision-Recall curves
ax = axes[0, 1]
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res["y_prob"])
    ax.plot(rec, prec, linewidth=2.5, label=f"{name} (AP={res['avg_precision']:.3f})",
            color=colors_map[name])
ax.axhline(y_test.mean(), color="k", linestyle="--", alpha=0.4, label=f"Baseline ({y_test.mean():.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 8c. Model comparison bar chart
ax = axes[0, 2]
names = list(results.keys())
short_names = ["LR", "RF", "GB"]
aucs = [results[n]["test_auc"] for n in names]
aps  = [results[n]["avg_precision"] for n in names]
x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, aucs, w, label="ROC-AUC", color="#4a90d9", edgecolor="white")
bars2 = ax.bar(x + w/2, aps,  w, label="Avg Precision", color="#e05c5c", edgecolor="white")
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(short_names)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title("Model Comparison", fontweight="bold")
ax.legend()

# 8d. Confusion matrix of best model
ax = axes[1, 0]
cm = confusion_matrix(y_test, best["y_pred"])
disp = ConfusionMatrixDisplay(cm, display_labels=["Retained", "Churned"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}", fontweight="bold")

# 8e. Feature importance (best model)
ax = axes[1, 1]
if hasattr(best["model"], "feature_importances_"):
    importances = best["model"].feature_importances_
    feat_names = X.columns.tolist()
    fi = pd.Series(importances, index=feat_names).sort_values(ascending=True).tail(12)
    colors = [PALETTE["churn"] if v > fi.mean() else PALETTE["neutral"] for v in fi.values]
    ax.barh(fi.index, fi.values, color=colors, edgecolor="white")
    ax.set_title(f"Feature Importances — {best_name}", fontweight="bold")
    ax.set_xlabel("Importance")
else:
    coef = pd.Series(np.abs(best["model"].coef_[0]), index=X.columns).sort_values(ascending=True).tail(12)
    ax.barh(coef.index, coef.values, color=PALETTE["neutral"], edgecolor="white")
    ax.set_title("Feature Coefficients (Abs) — LR", fontweight="bold")
    ax.set_xlabel("|Coefficient|")

# 8f. Churn probability distribution
ax = axes[1, 2]
ax.hist(best["y_prob"][y_test == 0], bins=30, alpha=0.7, color=PALETTE["retain"],
        label="Retained", edgecolor="white")
ax.hist(best["y_prob"][y_test == 1], bins=30, alpha=0.7, color=PALETTE["churn"],
        label="Churned", edgecolor="white")
ax.axvline(0.5, color="black", linestyle="--", alpha=0.6, label="Threshold 0.5")
ax.set_xlabel("Predicted Churn Probability"); ax.set_ylabel("Count")
ax.set_title(f"Score Distribution — {best_name}", fontweight="bold")
ax.legend()

plt.tight_layout()
fig4.savefig("/content/fig4_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n[✓] Figure 4 saved")


# ─────────────────────────────────────────────
# 9. BUSINESS INSIGHTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("BUSINESS INSIGHTS SUMMARY")
print("=" * 60)

churn_df = df_raw.copy()
high_risk = churn_df[churn_df["churn"] == 1]

print(f"""
Top Risk Factors:
  • Customers inactive for >60 days : {(churn_df['days_since_last_order'] > 60).mean():.1%} of base
  • Low satisfaction (score ≤ 2)    : {(churn_df['satisfaction_score'] <= 2).mean():.1%} of base
  • Tier 3 city customers           : churn rate {churn_df[churn_df['city_tier']==3]['churn'].mean():.1%}
  • High return rate (>20%)         : churn rate {churn_df[churn_df['return_rate']>0.2]['churn'].mean():.1%}

Model Performance (Best: {best_name}):
  • ROC-AUC              : {best['test_auc']:.4f}
  • Average Precision    : {best['avg_precision']:.4f}
  • Churners correctly caught: {confusion_matrix(y_test, best['y_pred'])[1,1]} / {(y_test==1).sum()}

Retention Recommendations:
  1. Re-engage customers inactive for 45+ days with personalized offers
  2. Follow up with customers who filed support tickets within 48 hours
  3. Launch app-adoption campaign for web-only customers
  4. Tier-3 city loyalty program — targeted discounts and faster delivery
  5. Flag satisfaction score ≤ 2 for immediate customer success outreach
""")

print("[✓] All outputs saved to /content/")


DATASET OVERVIEW
Shape          : (5000, 22)
Churn rate     : 7.38%
Missing values : 0

Data types:
int64      13
float64     6
object      3
Name: count, dtype: int64

First 3 rows:
   customer_id  tenure_months  age  gender  city_tier  num_orders  avg_order_value  total_spend  days_since_last_order  num_returns  return_rate  num_sessions  avg_session_duration_min  email_open_rate  app_used  support_tickets  satisfaction_score payment_method category_preference  discount_usage_rate  coupon_redeemed  churn
0            1             39   21  Female          2          21           175.15      3678.15                    112            1       0.0476           162                      1.45           0.1252         1                1                   1     Debit Card             Grocery               0.4993                1      0
1            2             52   57  Female          1          34            37.42      1272.28                     18            1       0.0294           199 